# AlexNet Benchmark

use PY310, py311 not support torch.compile, py39 not support libnvrtc.so compatibility

In [1]:
model_name = "vit-torch"

import torch
from torch import nn
from torchvision import models
#import torch_mlir
import numpy as np
import iree
import iree.compiler
import iree.runtime

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll


In [2]:
    
def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=33):
    return ti(stmt, globals=globals(), number=n) * 1000 / n

## Experimental

In [3]:
import torch
from torch import nn
from torchvision import models
import pandas as pd

MAGIC_NUM = 7777e-5

device = torch.device("cuda:0")
model = models.vit_b_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * MAGIC_NUM for k, v in model.state_dict().items()})
model = model.to(device)

df = pd.DataFrame()

### PyTorch (Baseline)

In [4]:
for bs in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]:
    model = torch.compile(model, backend="inductor")
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = model(image)
    grad = torch.randn_like(output)
    
    try:
        baseline_f = torch_model_benchmark(model, 
                                 [image], 
                                 device='gpu',
                                 warmups=0,
                                 repetitions=33, 
                                 measure_count=11)
        baseline_f = np.mean(baseline_f)
        baseline_b = torch_model_benchmark(torch.autograd.grad, 
                                 [output, [image], grad], 
                                 device='gpu',
                                 warmups=0,
                                 repetitions=33, 
                                 measure_count=11)
        baseline_b = np.mean(baseline_b)
        df = pd.concat([df, get_dataframe(baseline_f, baseline_b, "Torch Dynamo at batch-size = {}".format(bs))])
    except Exception as e:
        print(f"处理模型时出错，批量大小 {bs}: {e}")
        print(df)

处理模型时出错，批量大小 32: CUDA out of memory. Tried to allocate 74.00 MiB (GPU 0; 7.78 GiB total capacity; 7.23 GiB already allocated; 27.94 MiB free; 7.53 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF
         time      pass                             item
0    5.857762   Forward   Torch Dynamo at batch-size = 1
0   13.220981  Backward   Torch Dynamo at batch-size = 1
0   19.078743      Full   Torch Dynamo at batch-size = 1
0    9.509993   Forward   Torch Dynamo at batch-size = 2
0   21.443811  Backward   Torch Dynamo at batch-size = 2
0   30.953804      Full   Torch Dynamo at batch-size = 2
0   17.074988   Forward   Torch Dynamo at batch-size = 4
0   39.108634  Backward   Torch Dynamo at batch-size = 4
0   56.183622      Full   Torch Dynamo at batch-size = 4
0   30.752559   Forward   Torch Dynamo at batch-size = 8
0   71.171898  Backward   T

OutOfMemoryError: CUDA out of memory. Tried to allocate 148.00 MiB (GPU 0; 7.78 GiB total capacity; 7.26 GiB already allocated; 31.94 MiB free; 7.53 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

Process ForkProcess-25:
Process ForkProcess-28:
Process ForkProcess-8:
Process ForkProcess-7:
Process ForkProcess-4:
Process ForkProcess-30:
Process ForkProcess-10:
Process ForkProcess-17:
Process ForkProcess-13:
Process ForkProcess-5:
Process ForkProcess-22:
Process ForkProcess-27:
Process ForkProcess-6:
Process ForkProcess-23:
Process ForkProcess-21:
Process ForkProcess-18:
Process ForkProcess-19:
Process ForkProcess-31:
Process ForkProcess-29:
Process ForkProcess-2:
Process ForkProcess-24:
Process ForkProcess-26:
Process ForkProcess-11:
Process ForkProcess-32:
Process ForkProcess-16:
Process ForkProcess-20:
Process ForkProcess-12:
Process ForkProcess-1:
Process ForkProcess-9:
Traceback (most recent call last):
Process ForkProcess-3:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback

In [ ]:
df.style.hide(axis="index")

In [ ]:
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time Normalized (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")

In [ ]:
forward = df[df["pass"] == "Forward"]
forward["acceleration"] = baseline_f / forward["time"]
forward

In [ ]:
backward = df[df["pass"] == "Backward"]
backward["acceleration"] = baseline_b / backward["time"]
backward

In [ ]:
full = df[df["pass"] == "Full"]
full["acceleration"] = (baseline_b + baseline_f) / full["time"]
full

In [ ]:
df = pd.concat([forward, backward, full])
df.to_csv(f"{model_name}.csv")

sns.barplot(df, x="pass", y="acceleration", hue="item")
plt.xlabel("Pass")
plt.ylabel("Acceleration")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-acceleration.png")